# 🤖 Fraud Detection ML Notebook
Interactive notebook for training, testing, and experimenting with fraud detection models.

## Setup
Make sure you've installed the requirements:
```bash
pip install -r ../requirements.txt
```

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_curve, auc
import joblib

# Styling
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Libraries imported successfully!")

## 📊 Step 1: Load Data

In [ ]:
# Load dataset
df = pd.read_csv('../datasets/fraud_transactions.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nFraud Distribution:")
print(df['is_fraud'].value_counts())
print(f"\nFraud Rate: {df['is_fraud'].mean():.2%}")

df.head()

## 🔍 Step 2: Exploratory Data Analysis

In [ ]:
# Distribution of fraud vs normal transactions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Amount distribution
axes[0, 0].hist(df[df['is_fraud']==0]['amount'], bins=50, alpha=0.6, label='Normal', color='green')
axes[0, 0].hist(df[df['is_fraud']==1]['amount'], bins=50, alpha=0.6, label='Fraud', color='red')
axes[0, 0].set_xlabel('Amount')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Transaction Amount Distribution')
axes[0, 0].legend()

# Trust score distribution
axes[0, 1].hist(df[df['is_fraud']==0]['customer_trust_score'], bins=30, alpha=0.6, label='Normal', color='green')
axes[0, 1].hist(df[df['is_fraud']==1]['customer_trust_score'], bins=30, alpha=0.6, label='Fraud', color='red')
axes[0, 1].set_xlabel('Trust Score')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Customer Trust Score Distribution')
axes[0, 1].legend()

# Hour of day
hour_fraud = df[df['is_fraud']==1]['hour_of_day'].value_counts().sort_index()
hour_normal = df[df['is_fraud']==0]['hour_of_day'].value_counts().sort_index()
axes[1, 0].bar(hour_normal.index, hour_normal.values, alpha=0.6, label='Normal', color='green')
axes[1, 0].bar(hour_fraud.index, hour_fraud.values, alpha=0.6, label='Fraud', color='red')
axes[1, 0].set_xlabel('Hour of Day')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Transaction Hour Distribution')
axes[1, 0].legend()

# Transaction velocity
axes[1, 1].hist(df[df['is_fraud']==0]['transaction_velocity_1h'], bins=20, alpha=0.6, label='Normal', color='green')
axes[1, 1].hist(df[df['is_fraud']==1]['transaction_velocity_1h'], bins=20, alpha=0.6, label='Fraud', color='red')
axes[1, 1].set_xlabel('Transactions per Hour')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Transaction Velocity Distribution')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(10, 8))
correlation = df.corr()
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Correlation Heatmap')
plt.show()

## 🔧 Step 3: Prepare Data

In [ ]:
# Define features and target
features = [
    'amount',
    'customer_total_transactions',
    'customer_trust_score',
    'customer_average_transaction',
    'hour_of_day',
    'day_of_week',
    'transaction_velocity_1h',
    'location_distance_km'
]

X = df[features]
y = df['is_fraud']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Train set: {X_train.shape[0]} samples")
print(f"✅ Test set: {X_test.shape[0]} samples")

## 🎓 Step 4: Train Models

In [ ]:
# Train Logistic Regression
print("Training Logistic Regression...")
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
lr_score = lr_model.score(X_test_scaled, y_test)
print(f"✅ Accuracy: {lr_score:.2%}")

In [ ]:
# Train Random Forest
print("Training Random Forest...")
rf_model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf_model.fit(X_train_scaled, y_train)
rf_score = rf_model.score(X_test_scaled, y_test)
print(f"✅ Accuracy: {rf_score:.2%}")

## 📈 Step 5: Evaluate Models

In [ ]:
# ROC Curves
plt.figure(figsize=(10, 6))

# Logistic Regression ROC
lr_proba = lr_model.predict_proba(X_test_scaled)[:, 1]
lr_fpr, lr_tpr, _ = roc_curve(y_test, lr_proba)
lr_auc = auc(lr_fpr, lr_tpr)
plt.plot(lr_fpr, lr_tpr, label=f'Logistic Regression (AUC = {lr_auc:.3f})', linewidth=2)

# Random Forest ROC
rf_proba = rf_model.predict_proba(X_test_scaled)[:, 1]
rf_fpr, rf_tpr, _ = roc_curve(y_test, rf_proba)
rf_auc = auc(rf_fpr, rf_tpr)
plt.plot(rf_fpr, rf_tpr, label=f'Random Forest (AUC = {rf_auc:.3f})', linewidth=2)

# Diagonal line
plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')

plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Confusion Matrix for Random Forest
from sklearn.metrics import confusion_matrix

rf_pred = rf_model.predict(X_test_scaled)
cm = confusion_matrix(y_test, rf_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Random Forest')
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, rf_pred))

## 🎯 Step 6: Feature Importance

In [ ]:
# Feature importance from Random Forest
feature_importance = pd.DataFrame({
    'feature': features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance - Random Forest')
plt.gca().invert_yaxis()
plt.show()

print("\nFeature Importance Ranking:")
print(feature_importance)

## 💾 Step 7: Save Models

In [ ]:
# Save models
joblib.dump(lr_model, '../trained_models/logistic_model.pkl')
joblib.dump(rf_model, '../trained_models/rf_model.pkl')
joblib.dump(scaler, '../trained_models/scaler.pkl')

# Choose best model
best_model = rf_model if rf_auc > lr_auc else lr_model
joblib.dump(best_model, '../trained_models/best_model.pkl')

print("✅ Models saved successfully!")
print(f"   Best Model: {'Random Forest' if rf_auc > lr_auc else 'Logistic Regression'}")
print(f"   AUC Score: {max(rf_auc, lr_auc):.3f}")

## 🧪 Step 8: Test Predictions

In [ ]:
# Test with sample transactions
test_transactions = pd.DataFrame([
    {
        'amount': 50.0,
        'customer_total_transactions': 25,
        'customer_trust_score': 75.0,
        'customer_average_transaction': 45.0,
        'hour_of_day': 14,
        'day_of_week': 2,
        'transaction_velocity_1h': 1,
        'location_distance_km': 10.0,
        'label': 'Normal Transaction'
    },
    {
        'amount': 1500.0,
        'customer_total_transactions': 1,
        'customer_trust_score': 25.0,
        'customer_average_transaction': 30.0,
        'hour_of_day': 3,
        'day_of_week': 6,
        'transaction_velocity_1h': 10,
        'location_distance_km': 800.0,
        'label': 'Suspicious Transaction'
    }
])

# Predict
X_test_sample = test_transactions[features]
X_test_sample_scaled = scaler.transform(X_test_sample)
predictions = best_model.predict_proba(X_test_sample_scaled)[:, 1]

# Display results
test_transactions['fraud_score'] = (predictions * 100).astype(int)
test_transactions['prediction'] = ['FRAUD ⚠️' if p > 0.6 else 'SAFE ✅' for p in predictions]

print("\n🧪 Test Predictions:")
print(test_transactions[['label', 'fraud_score', 'prediction']])

## ✅ Next Steps

1. **Deploy Models**: Copy trained models to production
2. **Monitor Performance**: Track accuracy over time
3. **Retrain Regularly**: Use new transaction data
4. **A/B Testing**: Compare with rule-based system

Ready to deploy? Run `python ../train.py` to save production-ready models!